# 11 - Governanca, Fairness e Limitacoes

Este notebook implementa a tarefa 25 da ordem recomendada: fazer analise de governanca e fairness com metricas por subgrupo.

Objetivos:

1. Avaliar representatividade por classe, sexo, idade e posicao da imagem.
2. Rodar predicoes do modelo final no conjunto de teste.
3. Calcular metricas por subgrupo.
4. Medir gaps entre subgrupos.
5. Gerar texto tecnico para o relatorio final.

Este notebook nao prova seguranca clinica. Ele documenta riscos, limitacoes e evidencias iniciais para governanca academica.

## Pre-requisitos

Execute antes:

1. `02_preprocessamento_splits.ipynb`
2. Notebooks de treinamento dos modelos.
3. `09_comparacao_modelos.ipynb`, para exportar o modelo final em `models/exported/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataset, create_dataloader
from src.inference import load_model_from_checkpoint
from src.training.fairness import (
    add_age_group,
    collect_predictions_with_metadata,
    merge_predictions_with_split_metadata,
    representation_table,
    subgroup_gaps,
    subgroup_metric_table,
)

config.ensure_project_directories()

if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

TEST_SPLIT_PATH = config.SPLITS_DIR / "test.csv"
print("DEVICE:", config.DEVICE)
print("TEST_SPLIT_PATH:", TEST_SPLIT_PATH)
print("EXPORTED_MODELS_DIR:", config.EXPORTED_MODELS_DIR)

## Representatividade no conjunto de teste

Antes de avaliar fairness do modelo, analisamos se o conjunto de teste possui subgrupos suficientes. Subgrupos pequenos devem ser interpretados com cautela.

In [ ]:
if not TEST_SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Split de teste nao encontrado: {TEST_SPLIT_PATH}. "
        "Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

test_df = pd.read_csv(TEST_SPLIT_PATH)
test_df = add_age_group(test_df)

representation_tables = []
for group_col in ["binary_label_name", "Patient Gender", "View Position", "age_group"]:
    table = representation_table(test_df, group_col=group_col)
    table.insert(0, "analysis_group", group_col)
    representation_tables.append(table)

representation_df = pd.concat(representation_tables, ignore_index=True)
representation_path = config.TABLES_DIR / "fairness_representatividade_teste.csv"
representation_df.to_csv(representation_path, index=False)
representation_df.head(20)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
summary_specs = [
    ("Patient Gender", "Distribuicao por sexo"),
    ("View Position", "Distribuicao por posicao"),
    ("age_group", "Distribuicao por faixa etaria"),
]

for ax, (column, title) in zip(axes, summary_specs):
    counts = test_df[column].astype("string").fillna("Unknown").value_counts().reset_index()
    counts.columns = [column, "image_count"]
    ax.bar(counts[column], counts["image_count"], color="#4C78A8")
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel("Quantidade")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "fairness_representatividade_subgrupos.png", dpi=160, bbox_inches="tight")
plt.show()

## Predicoes do modelo final

Carregamos o checkpoint exportado e executamos predicao no conjunto de teste com metadados.

In [ ]:
model, model_metadata = load_model_from_checkpoint(device=config.DEVICE)
model_metadata

In [ ]:
test_dataset = create_dataset(
    split_name="test",
    split_csv=TEST_SPLIT_PATH,
    return_metadata=True,
)
test_loader = create_dataloader(
    dataset=test_dataset,
    split_name="test",
    dataloader_config=DataLoaderConfig(
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    ),
)

predictions = collect_predictions_with_metadata(
    model=model,
    dataloader=test_loader,
    device=config.DEVICE,
)
predictions = merge_predictions_with_split_metadata(predictions, TEST_SPLIT_PATH)
predictions_path = config.METRICS_DIR / "fairness_test_predictions.csv"
predictions.to_csv(predictions_path, index=False)
predictions.head()

## Metricas por subgrupo

Calculamos accuracy, precision, recall, F1 e AUC por sexo, faixa etaria e posicao da imagem. O minimo padrao e 10 amostras por subgrupo.

In [ ]:
MIN_SUBGROUP_SAMPLES = 10
subgroup_metric_tables = []

for group_col in ["Patient Gender", "age_group", "View Position"]:
    table = subgroup_metric_table(
        predictions=predictions,
        group_col=group_col,
        min_samples=MIN_SUBGROUP_SAMPLES,
    )
    subgroup_metric_tables.append(table)

subgroup_metrics_df = pd.concat(subgroup_metric_tables, ignore_index=True)
subgroup_metrics_path = config.METRICS_DIR / "fairness_metricas_por_subgrupo.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
subgroup_metrics_df

In [ ]:
gap_df = subgroup_gaps(subgroup_metrics_df)
gap_path = config.METRICS_DIR / "fairness_gaps_por_subgrupo.csv"
gap_df.to_csv(gap_path, index=False)
gap_df

## Graficos de fairness

Os graficos abaixo destacam recall e F1 por subgrupo. Recall recebe atencao especial porque falsos negativos em saude sao criticos.

In [ ]:
for metric in ["recall", "f1", "precision"]:
    plot_data = subgroup_metrics_df[subgroup_metrics_df["skipped_low_sample"] == False].copy()
    if plot_data.empty:
        print(f"Sem subgrupos suficientes para grafico de {metric}.")
        continue

    fig, ax = plt.subplots(figsize=(12, 6))
    labels = plot_data["group_column"] + ": " + plot_data["group_value"]
    ax.bar(labels, plot_data[metric], color="#59A14F")
    ax.set_title(f"{metric.upper()} por subgrupo")
    ax.set_xlabel("Subgrupo")
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    fig.savefig(config.FIGURES_DIR / f"fairness_{metric}_por_subgrupo.png", dpi=160, bbox_inches="tight")
    plt.show()

## Discussao para o relatorio

A celula abaixo gera um texto inicial para `reports/fairness_discussao.md`.

In [ ]:
largest_gaps = gap_df.dropna(subset=["gap"]).sort_values("gap", ascending=False).head(8)

discussion = f"""
# Governanca, Fairness e Limitacoes

A analise de governanca foi realizada sobre o conjunto de teste e considera os subgrupos disponiveis no dataset: sexo do paciente, faixa etaria e posicao da imagem (`PA`/`AP`). O modelo avaliado foi `{model_metadata['model_name']}`.

## Representatividade

Foram geradas tabelas de representatividade por classe, sexo, idade e posicao da imagem. Subgrupos com baixa quantidade de amostras foram marcados para interpretacao cautelosa, pois metricas calculadas com poucos exemplos podem oscilar bastante.

## Metricas por subgrupo

Foram calculadas accuracy, precision, recall, F1-score e AUC-ROC por subgrupo. O recall deve receber atencao especial porque falsos negativos em um contexto de saude podem atrasar investigacao clinica. As maiores diferencas observadas entre subgrupos estao registradas em `reports/metricas/fairness_gaps_por_subgrupo.csv`.

## Principais gaps observados

{largest_gaps.to_markdown(index=False) if not largest_gaps.empty else 'Nao foi possivel calcular gaps com os dados disponiveis.'}

## Riscos clinicos e limites de uso

Este prototipo nao substitui avaliacao medica. Os labels do NIH Chest X-rays podem conter ruido, pois foram derivados de laudos e podem refletir limitacoes do processo de anotacao. Falsos negativos podem atrasar uma investigacao clinica, enquanto falsos positivos podem gerar ansiedade ou exames adicionais desnecessarios. Antes de qualquer uso real, seriam necessarias validacao clinica, revisao por especialistas, avaliacao prospectiva, auditoria de vieses e governanca de dados.
""".strip()

discussion_path = config.REPORTS_DIR / "fairness_discussao.md"
discussion_path.write_text(discussion + "\n")
print(discussion)
print("\nDiscussao salva em:", discussion_path)

## Arquivos gerados

- `reports/tabelas/fairness_representatividade_teste.csv`
- `reports/metricas/fairness_test_predictions.csv`
- `reports/metricas/fairness_metricas_por_subgrupo.csv`
- `reports/metricas/fairness_gaps_por_subgrupo.csv`
- `reports/figuras/fairness_representatividade_subgrupos.png`
- `reports/figuras/fairness_recall_por_subgrupo.png`
- `reports/figuras/fairness_f1_por_subgrupo.png`
- `reports/figuras/fairness_precision_por_subgrupo.png`
- `reports/fairness_discussao.md`